# Meet Two (and a half) Buoyant Plume Theory Models!

### This notebook provides some executable examples of code to run the **full**, **analytical**, and **modified analytical** models described in the Intro to BPT document. It includes some built-in sliders and other interactive elements that you can use to tweak certain variables used in the models and see how the outputs change. 

### **NOTE: Changes made in the Binder session are not saved. If you want to keep your edits, download the notebook from within the Binder session before closing the tab.** 

#### The python scripts containing the code for the full and analytical models are located in the **plume_model_code/** directory. The **full_plume.py** and **analytical_plume_gen.py** scripts were adapted into python by Natalie McGee from existing Matlab scripts by Donald Slater and Aurora Roth, respectively.

In [10]:
import numpy as np
import scipy
import matplotlib.pyplot as plt
import gsw
import ipywidgets as widgets
from ipywidgets import interact, IntRangeSlider, Layout, RadioButtons
from IPython.display import display
import pandas as pd
from plume_model_code.full_plume import run_plume
from plume_model_code.analytical_plume_gen import analytical_plume
import warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)


# Full Model: Depth Space #

#### This model is for a **line-source plume** with the specific application to ice-ocean interactions, as described in the accompanying document. This notebook allows you to get an idea of the parameters we can control and how they affect the model. 

#### In this module, you can see an example of the plume model output in depth-space. It requires a vertical profile of temperature and salinity data for the **ambient water being entrained** into the plume, and you can test the model with a uniform/unstratified ambient water mass, a linearly changing ambient profile, or a real CTD profile from Greenland. ####

### **About the variables:** 
#### **Profile:** select the stratification profile (temperature and salinity) of the ambient water 
#### **Q0:** (m$^2$/s) the volume flux per unit width at the plume source
#### **alpha:** (unitless) the value of the entrainment constant relating entrainment rate to plume velocity


### **For more details see the associated Intro to BPT document.**

In [11]:
linear_data = "../example_profiles/linear.csv"
unstratified_data = "../example_profiles/unstratified.csv"
real_data = "../example_profiles/real_profile.csv"
   
plume_depth = 700   # For this example we are using a grounding depth of 700 m

@interact(
    profile=RadioButtons(
        options=['Unstratified', 'Linear', 'Real Profile'],
        description='Ambient Stratification:',
        style={'description_width': 'initial'}),
    Q0=(0.05, 1.0, 0.05),
    alpha=(0.05, 0.2, 0.01)
)

def plot_profile(profile, Q0, alpha):

    if profile == "Unstratified":
        data = unstratified_data

    elif profile == "Linear":
        data = linear_data

    elif profile == "Real Profile":
        data = real_data
    
    # Load the CSV into a DataFrame
    df = pd.read_csv(data)
    
    # Convert ambient profile into numpy arrays
    depth = np.array(df["Depth"])
    salinity = np.array(df["Absolute Salinity"])
    temp = np.array(df["Temperature"])
    
    zi = -depth[::-1]   # Depth must be negative and ust input values from deepest to shallowest!
    Ta = temp[::-1]     # Ambient fjord temperature at zi
    Sa = salinity[::-1] # Ambient fjord salinity at zi
    
    xi = np.zeros_like(zi)    # Sets the shape of the glacial front. Currently vertical
    Na = np.ones_like(zi)*10  # Ambient fjord nitrate at zi. (If you are interested in nitrate)
    
    alpha = alpha # entrainment coefficient
    Q0 = Q0  # Subglacial volume flux (m2/s)
    
    # Run the plume model!
    sol=run_plume(zi, xi, Ta, Sa, Na, Q0, alpha)
    
    # Set up the figure
    fig, axes = plt.subplots(1, 5, figsize=(13, 5), sharey= True)
    
    # First panel: Plume width
    axes[0].plot(sol["b"], sol['z'], color = "darkblue", label = "Model")
    axes[0].hlines(sol['zNB'], 0, 300, color = "black", linestyle = "dotted", label = "NBD")
    axes[0].set_ylim(-700, 0)
    axes[0].set_xlim(0, 300)
    axes[0].set_ylabel("Depth [m]")
    axes[0].set_xlabel("Plume Width" "\n[m]", fontsize = 10)
    axes[0].legend(loc="lower right")

    # Second panel: Plume salinity
    axes[1].plot(sol["S"], sol['z'], color = "darkblue", label = "Model")
    axes[1].plot(Sa, zi, color = "red", linewidth = 1, linestyle = "dashed")
    axes[1].set_ylim(-700, 0)
    axes[1].set_xlim(30, 35)
    axes[1].set_xlabel("Absolute Salinity" "\n[g/kg]", fontsize = 10)

    # Third panel: Plume temperature
    axes[2].plot(sol["T"], sol['z'], color = "darkblue", label = "Model")
    axes[2].plot(Ta, zi, color = "red", linewidth = 1, linestyle = "dashed", label = "Ambient")
    axes[2].set_ylim(-700, 0)
    axes[2].set_xlim(-1, 3)
    axes[2].set_xlabel("Conservative Temp." "\n[°C]", fontsize = 10)
    axes[2].legend(loc=(0.03, 0.06))

    # Fourth panel: Plume vertical velocity
    axes[3].plot(sol["u"], sol['z'], color = "darkblue")
    axes[3].set_ylim(-700, 0)
    axes[3].set_xlim(0, 1.8)
    axes[3].set_xlabel("Velocity" "\n[m/s]", fontsize = 10)

    # Fifth panel: Melt rate along glacier front
    axes[4].plot(sol["mdot"], sol['z'], color = "darkblue")
    axes[4].set_ylim(-700, 0)
    axes[4].set_xlim(0, 4)
    axes[4].set_xlabel(r"Terminus Melt Rate" "\n[m$^2$/day]", fontsize = 10)

    plt.show()
    

interactive(children=(RadioButtons(description='Ambient Stratification:', options=('Unstratified', 'Linear', '…

#### (You can ignore any runtime warnings that may appear)

#### **NBD is the plume's "Neutral Buoyancy Depth"** or intrusion depth, where it has the same density as the ambient water. Note that a plume rising through an **unstratified** ambient water mass will **never** reach neutral buoyancy below the surface. 

# Compare the Full and Analytical Models: TS Space 

#### It can be useful to see where the model predicts the temperature and salinity of the plume will fall in T-S (temperature-salinity) space.

#### This module compares the predictions made by the **full model**, the **analytical model**, and the **modified analytical model**. (For descriptions of these, see the Intro to BPT document). 

#### While the full model can use an arbitrary profile of ambient temperature and salinity data, the analytical model assumes a **uniform** ambient water mass made up of deep **"Atlantic Water" (AW).** With the sliders below, you can change how that **Atlantic Water mass** is defined and see how the definition impacts the analytical models.

#### Finally, the **modified analytical model** includes a 41% correction using polar water (closer to the surface). You can experiment with changing the **definition of PW** as well to see how it affects the predictions made by the modified analytical model. 

In [12]:
real_data = "../example_profiles/real_profile.csv"
   
plume_depth = 700   # For this example we are using a grounding depth of 700 m

@interact(
    DW_range = IntRangeSlider(
        value=[-700, -650],
        min=-700,
        max=-500,
        step=10,
        description='AW Definition [m]',
        style={'description_width': 'initial'},
        layout=Layout(width='400px')),
    PW_range = IntRangeSlider(
        value=[-200, -175],
        min=-300,
        max=-50,
        step=5,
        description='PW Definition [m]',
        style={'description_width': 'initial'},
        layout=Layout(width='400px')),
    Q0=(0.05, 1.0, 0.05),
    alpha=(0.05, 0.2, 0.01)
)

def plot_profile(DW_range, PW_range, Q0, alpha):

    data = real_data
    
    # Load the CSV into a DataFrame
    df = pd.read_csv(data)
    
    # Convert ambient profile into numpy arrays
    depth = np.array(df["Depth"])
    salinity = np.array(df["Absolute Salinity"])
    temp = np.array(df["Temperature"])
    
    zi = -depth[::-1]   # Depth must be negative and ust input values from deepest to shallowest!
    Ta = temp[::-1]     # Ambient fjord temperature at zi
    Sa = salinity[::-1] # Ambient fjord salinity at zi
    
    xi = np.zeros_like(zi)    # Sets the shape of the glacial front. Currently vertical
    Na = np.ones_like(zi)*10  # Ambient fjord nitrate at zi. (If you are interested in nitrate)
    
    alpha = alpha # entrainment coefficient
    Q0 = Q0  # Subglacial volume flux (m2/s)

    #-----------------------------
    # Full plume model
    #-----------------------------
    
    sol=run_plume(zi, xi, Ta, Sa, Na, Q0, alpha)

    #-----------------------------
    # Analytical plume model
    #-----------------------------

    DW_lower_bound = 700+DW_range[0]
    DW_upper_bound = 700+DW_range[1]

    S_DW=np.average(Sa[DW_lower_bound:DW_upper_bound]) # Ambient properties: average the first ?m above the plume depth
    T_DW=np.average(Ta[DW_lower_bound:DW_upper_bound])
    Tplume, Splume, AWp, SGDp, SMWp, Q_AW, Q_SMW = analytical_plume(T_AW=T_DW,
                                                                    S_AW=S_DW,
                                                                    Q_SGD=Q0,
                                                                    h_gl=plume_depth,
                                                                    w=1,
                                                                    alpha=alpha)
    #-----------------------------
    # Modified Analytical plume model
    #-----------------------------

    PW_lower_bound = 700+PW_range[0]
    PW_upper_bound = 700+PW_range[1]

    S_PW=np.average(Sa[PW_lower_bound:PW_upper_bound]) # Ambient properties: average the first ?m above the plume depth
    T_PW=np.average(Ta[PW_lower_bound:PW_upper_bound])
    
    S_mod = Splume*0.59 + S_PW*0.41
    T_mod = Tplume*0.59 + T_PW*0.41

    
    # Set up the figure
    fig, axes = plt.subplots(1, 1, figsize=(6, 5))
    
    axes.scatter(Sa, Ta, color = 'red', label = "Ambient", s=2)
    axes.scatter(sol["SNB"], sol['TNB'], color = "darkblue", edgecolors= "black", label = "Full Model")
    axes.scatter(Splume, Tplume, color = 'gold', edgecolors= "black", label = "Analytical Model")
    axes.scatter(S_mod, T_mod, color = 'darkgoldenrod', edgecolors= "black", label = "Modified Analytical Model")
    axes.scatter(S_DW, T_DW, color = "darkred", marker = "D", label = "Atlantic Water")
    axes.scatter(S_PW, T_PW, color = "blue", marker = "D", label = "Polar Water")

    axes.set_xlim(33.5, 34.8)
    axes.set_ylabel("Conservative Temp [°C]")
    axes.set_xlabel("Absolute Salinity [g/kg]", fontsize = 10)
    axes.legend(loc="lower right")

    axes.set_title("Final plume T, S predicted by different models")

    plt.show()

interactive(children=(IntRangeSlider(value=(-700, -650), description='AW Definition [m]', layout=Layout(width=…

#### AW and PW properties are defined by averaging the ambient temperature and salinity over the range of depths defined by the sliders. 